# HashMM-RAG quickstart on AutoDL

This notebook walks the full pipeline end-to-end. It mirrors `scripts/00`–`05`
but lets you stop and inspect after each step.

Prerequisites:
- ran `pip install -e ".[hash,ingest,eval]"` once
- copied `.env.example` → `.env` and filled `LLM_API_KEY`
- `HF_ENDPOINT=https://hf-mirror.com` set in env (already in `.env.example`)

In [ ]:
# Step 0 — bootstrap a few arXiv PDFs (~10 MB total)
!python scripts/00_download_arxiv.py --output /root/autodl-tmp/hashmm/data/pdfs

In [ ]:
# Step 1 — parse all PDFs with RAG-Anything (slow first run while MinerU bootstraps)
!python scripts/01_parse_documents.py \
    --input  /root/autodl-tmp/hashmm/data/pdfs \
    --output /root/autodl-tmp/hashmm/data/parsed

In [ ]:
# Inspect: what does a parsed JSON look like?
import json
from pathlib import Path
first = next(Path('/root/autodl-tmp/hashmm/data/parsed').glob('*.json'))
d = json.loads(first.read_text())
print('doc_id   :', d['doc_id'][:30], '…')
print('items    :', len(d['content_list']))
from collections import Counter
print('by type  :', Counter(it.get('type','?') for it in d['content_list']))
# Show first non-text item (probably an image with caption)
for it in d['content_list']:
    if it.get('type') != 'text':
        print('\nfirst non-text item:')
        print(json.dumps(it, indent=2, ensure_ascii=False)[:600])
        break

In [ ]:
# Step 2 — extract chunks + cross-modal pairs
!python scripts/02_extract_pairs.py \
    --parsed /root/autodl-tmp/hashmm/data/parsed \
    --chunks /root/autodl-tmp/hashmm/data/chunks.jsonl \
    --pairs  /root/autodl-tmp/hashmm/data/pairs.jsonl

In [ ]:
# Inspect: how many pairs from which source?
from collections import Counter
from hashmm.ingestion.chunk_extractor import read_pairs_jsonl
pairs = read_pairs_jsonl('/root/autodl-tmp/hashmm/data/pairs.jsonl')
print(f'total pairs: {len(pairs)}')
print('by source  :', Counter(p.source for p in pairs))
print('\nexample pairs:')
for p in pairs[:5]:
    print(f'  [{p.source:15s}] {p.text[:80]!r} <-> {p.image_path}')

In [ ]:
# Step 3 — train the hash net. Default 20 epochs.
# Watch the val mAP@10 climb; we want both t2i and i2t > 0.5 after a few epochs.
!python scripts/03_train_hash_net.py \
    --pairs /root/autodl-tmp/hashmm/data/pairs.jsonl

In [ ]:
# Step 4 — encode all chunks and build the Faiss binary index
!python scripts/04_build_index.py \
    --chunks /root/autodl-tmp/hashmm/data/chunks.jsonl

In [ ]:
# Step 5 — try some queries
from hashmm.config import HashMMConfig
from hashmm.retrieval.hash_retriever import HashRetriever

cfg = HashMMConfig()
retriever = HashRetriever(cfg)   # lazy-loads net + encoders + index on first call

for q in [
    'cross-modal retrieval architecture',
    'deep hashing for image retrieval',
    'late interaction multi-vector model',
]:
    print(f'\n=== Query: {q!r} ===')
    for r in retriever.retrieve(q, top_k=5):
        ham = r.meta.get('hamming_dist','')
        snippet = (r.text or '')[:100]
        print(f'  ham={ham:3} [{r.modality:8s}] {snippet}')

In [ ]:
# Cross-modal: text query, but filter to image hits only
for r in retriever.retrieve('figure of system architecture', top_k=10, modality_hint='image'):
    print(f'  ham={r.meta.get("hamming_dist")}  image={r.image_path}')

## What to do next

1. **Inspect val mAP**: in the training log, check `val_map_t2i` and `val_map_i2t`. If they're below 0.3 after 20 epochs your training set is too small — drop in more PDFs.
2. **Sweep bits**: try `HASH_BITS=64` and `HASH_BITS=256` in `.env`, retrain, compare. This is your first benchmark plot.
3. **Compare against vector retrieval**: once RAG-Anything's LightRAG side is also populated (run `process_document_complete` end-to-end), use `HybridRouter` with both retrievers and run side-by-side.
4. **Then we add the agent layer (M4) on top.**